# Tratando os dados dp PgAdmin

- @author: Guilherme Nogueira

## Importações

In [1]:
# =========================================================
# BIBLIOTECAS
# =========================================================
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import text
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError

from io import BytesIO
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

from sqlalchemy import text


# =========================================================
# FUNÇÕES
# =========================================================
import sys
from pathlib import Path

PASTA_FUNCOES = Path(r"C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-functions\functions")

if not PASTA_FUNCOES.exists():
    raise FileNotFoundError(
        f"Pasta das funções não encontrada:\n{PASTA_FUNCOES}"
    )

if str(PASTA_FUNCOES) not in sys.path:
    sys.path.insert(0, str(PASTA_FUNCOES))

from excel_format import (
    exportar_xlsx_formatado,
    exportar_varias_abas_xlsx,
)

print("Funções de Excel importadas com sucesso.")

Funções de Excel importadas com sucesso.


### Configurações de conexão

In [2]:
# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações de conexão
PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_SCHEMA = os.getenv("PG_SCHEMA", "analytics_mart")

# Verifica se as configurações obrigatórias foram preenchidas
configuracoes = {
    "PG_HOST": PG_HOST,
    "PG_DATABASE": PG_DATABASE,
    "PG_USER": PG_USER,
    "PG_PASSWORD": PG_PASSWORD,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if valor is None or str(valor).strip() == ""
]

if faltantes:
    raise ValueError(
        "As seguintes variáveis não foram preenchidas no arquivo .env: "
        + ", ".join(faltantes)
    )


# Criação segura da URL de conexão
url_conexao = URL.create(
    drivername="postgresql+psycopg",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
    database=PG_DATABASE,
)


# Engine de conexão
engine = create_engine(
    url_conexao,
    pool_pre_ping=True,
)

### Testar a conexão

In [3]:
try:
    with engine.connect() as conexao:
        resultado = conexao.execute(
            text(
                """
                SELECT
                    current_database() AS banco,
                    current_user AS usuario,
                    current_schema() AS schema_atual,
                    version() AS versao
                """
            )
        ).mappings().one()
    
    print("Conexão realizada com sucesso!")
    print(f"Banco: {resultado['banco']}")
    print(f"Usuário: {resultado['usuario']}")
    print(f"Schema atual: {resultado['schema_atual']}")

except SQLAlchemyError as erro:
    print("Não foi possível conectar ao PostgreSQL.")
    raise erro

Conexão realizada com sucesso!
Banco: postgres
Usuário: lradmin
Schema atual: public


### Configuração das Views

In [80]:
# ============================================================
# CONFIGURAÇÕES DAS VIEWS
# ============================================================

# Chaves usadas nos merges
CHAVES = ["id_property", "reference_month"]

# Período analisado
DATA_INICIAL = pd.Timestamp("2025-01-01")

# Primeiro dia do mês atual
DATA_FINAL = (pd.Timestamp.today().to_period("M").to_timestamp())

# View principal da tabela final
VIEW_BASE = "vw_revenue"

# Views que serão adicionadas à view principal
VIEWS_MERGE = [
    "vw_cattle",
    "vw_expense",
    "vw_feeding",
    "vw_labor",
    "vw_own_milk",
    "vw_area_month",
    "vw_asset_payment_history",
]

# Lista completa de views autorizadas para importação
views = list(dict.fromkeys( [VIEW_BASE] + VIEWS_MERGE ))

# ============================================================
# CONFIGURAÇÃO ESPECÍFICA DA ÁREA ATIVA
# ============================================================

NOME_VIEW_AREA = "vw_area_month"

COLUNAS_AREA_ATIVA = [
    "month_start",
    "id_property",
    "hectares_owned_mes",
    "hectares_rented_mes",
    "hectares_total_mes",
    "raw_land_value_avg_weighted_mes",
]

# ============================================================
# FUNÇÃO DE IMPORTAÇÃO
# ============================================================

def importar_view(nome_view: str, engine, schema: str, ordenar_por: str | None = None,) -> pd.DataFrame:
    """
    Importa uma view PostgreSQL para um DataFrame.

    A view precisa estar cadastrada na lista `views`. 
    A ordenação é feita no pandas somente quando a coluna informada existir.
    """

    if nome_view not in views:
        raise ValueError( f"View não autorizada: {nome_view}" )

    print(f"Importando {schema}.{nome_view}...")
    
    consulta = text(f''' SELECT * FROM "{schema}"."{nome_view}"; ''')
    
    df = pd.read_sql_query(sql=consulta, con=engine)

    if (ordenar_por is not None and ordenar_por in df.columns):
        df = (df.sort_values(ordenar_por).reset_index(drop=True) )

    return df


# ============================================================
# CONFERÊNCIA DAS CONFIGURAÇÕES
# ============================================================

print(f"View principal: {VIEW_BASE}")

print("\nViews usadas nos merges:")
for nome_view in VIEWS_MERGE:
    print(f"- {nome_view}")

print("\nViews autorizadas para importação:")
for nome_view in views:
    print(f"- {nome_view}")

print(f"\nPeríodo: " f"{DATA_INICIAL:%Y-%m} a {DATA_FINAL:%Y-%m}" )

View principal: vw_revenue

Views usadas nos merges:
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- vw_area_month
- vw_asset_payment_history

Views autorizadas para importação:
- vw_revenue
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- vw_area_month
- vw_asset_payment_history

Período: 2025-01 a 2026-07


### Criar Driver Supabase

In [28]:
from supabase import create_client, Client

service_key = os.getenv('SUPABASE_SERVICE_KEY')
print("Chave carregada?", service_key is not None)

# URL do Projeto
project_url = 'https://mrjrkkbecjyzzwkvouxx.supabase.co'

# Acesso ao cliente
global supabase
supabase: Client = create_client(project_url, service_key)
print("Supabase conectado!")

Chave carregada? True
Supabase conectado!


## IGPDI

Ações básicas da função abaixo:
1. Verifica se há recente atualização do IGPDI que não está na base.
2. Se houver, baixa nova planilha, trata e cria as colunas e realiza o upload no supabase.
3. Se não houver, traz a base de dados do supabase

In [10]:
URL_IGPDI = "https://sindusconpr.com.br/igp-di-fgv-308-p/"

HEADERS_IGPDI = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
    )
}


def localizar_link_igpdi(timeout=30):
    """
    Acessa a página do Sinduscon-PR e localiza automaticamente o link de download da série histórica do IGP-DI.
    """
    resposta = requests.get(
        URL_IGPDI,
        headers=HEADERS_IGPDI,
        timeout=timeout
    )
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "html.parser")

    # Busca prioritária: botão DOWNLOAD dentro da linha do IGP-DI
    for link in soup.select("a[href]"):
        texto_link = link.get_text(" ", strip=True).upper()

        linha = link.find_parent("tr")
        contexto = (
            linha.get_text(" ", strip=True).upper()
            if linha is not None
            else link.parent.get_text(" ", strip=True).upper()
        )

        if "DOWNLOAD" in texto_link and "IGP" in contexto:
            return urljoin(URL_IGPDI, link["href"])

    # Busca alternativa pelos links de download da página
    for link in soup.select("a[href]"):
        href = link.get("href", "")

        if "/download/" in href:
            return urljoin(URL_IGPDI, href)

    raise RuntimeError(
        "Não foi possível localizar o arquivo XLSX do IGP-DI na página."
    )


def carregar_igpdi_supabase():
    """
    Carrega a tabela atual do IGP-DI armazenada no Supabase.
    """
    response = (
        supabase
        .table("tab_igpdi")
        .select("data,igpdi,igpdi_atual,deflator")
        .execute()
    )

    df = pd.DataFrame(response.data or [])

    if df.empty:
        return df

    df["data"] = pd.to_datetime(df["data"], errors="coerce")

    for coluna in ["igpdi", "igpdi_atual", "deflator"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    return (
        df
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )


def series_igpdi_iguais(df_site, df_supabase):
    """
    Verifica se a série do site é igual à série do Supabase.
    """
    if df_site.empty or df_supabase.empty:
        return False

    site = (
        df_site[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    supa = (
        df_supabase[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    if len(site) != len(supa):
        return False

    mesmas_datas = site["data"].equals(supa["data"])

    mesmos_valores = np.allclose(
        site["igpdi"].to_numpy(dtype=float),
        supa["igpdi"].to_numpy(dtype=float),
        equal_nan=True
    )

    return mesmas_datas and mesmos_valores


def baixar_dados_igpdi(timeout=30):
    """
    Baixa a planilha do IGP-DI diretamente, sem Selenium.

    Se a série estiver igual à armazenada no Supabase, retorna os dados do Supabase.

    Se houver alteração, recalcula o deflator e atualiza toda a tabela no Supabase.
    """
    # 1. Localizar o arquivo
    link_download = localizar_link_igpdi(timeout=timeout)

    # 2. Baixar o XLSX diretamente
    resposta = requests.get(
        link_download,
        headers={
            **HEADERS_IGPDI,
            "Referer": URL_IGPDI
        },
        timeout=timeout
    )
    resposta.raise_for_status()

    # Arquivos XLSX são arquivos ZIP e normalmente começam com PK
    if not resposta.content.startswith(b"PK"):
        raise RuntimeError(
            "O conteúdo baixado não parece ser um arquivo XLSX válido."
        )

    # 3. Ler e tratar sem salvar na pasta Downloads
    arquivo_memoria = BytesIO(resposta.content)

    # A Plan1 possui três linhas de título antes da série histórica.
    df_site = pd.read_excel(
        arquivo_memoria,
        sheet_name="Plan1",
        header=None,
        skiprows=3,
        usecols=[0, 1],
        names=["data", "igpdi"],
    )

    df_site["data"] = pd.to_datetime(df_site["data"], errors="coerce")
    df_site["igpdi"] = pd.to_numeric(df_site["igpdi"], errors="coerce")

    df_site = (
        df_site
        .dropna(subset=["data", "igpdi"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if df_site.empty:
        raise ValueError("Nenhum registro válido de IGP-DI foi encontrado na planilha.")

    # Mantém a convenção existente: deflator = índice do mês / índice mais recente.
    igpdi_atual = df_site["igpdi"].iloc[-1]
    df_site["igpdi_atual"] = igpdi_atual
    df_site["deflator"] = df_site["igpdi"] / igpdi_atual

    # 4. Consultar Supabase
    try:
        df_supabase = carregar_igpdi_supabase()
    except Exception as erro:
        print(f"⚠️ Não foi possível consultar o Supabase: {erro}")
        df_supabase = pd.DataFrame()

    # 5. Retornar Supabase se não houver alteração
    if series_igpdi_iguais(df_site, df_supabase):
        print(f"✅ IGP-DI já está atualizado no Supabase. Último mês: {df_supabase['data'].max():%m/%Y}" )

        return df_supabase

    # 6. Preparar os registros para envio
    df_upload = df_site.copy()
    df_upload["data"] = df_upload["data"].dt.strftime("%Y-%m-%d")
    df_upload = ( df_upload .astype(object) .where(pd.notna(df_upload), None) )

    registros = df_upload.to_dict("records")

    # 7. Atualizar toda a tabela porque o deflator histórico muda
    try:
        (supabase.table("tab_igpdi").delete().gte("data", "1900-01-01").execute())
        (supabase .table("tab_igpdi").insert(registros).execute())
        print(f"✅ Supabase atualizado com {len(registros)} registros. Último mês: {df_site['data'].max():%m/%Y}")

    except Exception as erro:
        raise RuntimeError( f"Erro ao atualizar a tabela tab_igpdi: {erro}" ) from erro

    return df_site

In [11]:
df_igpdi = baixar_dados_igpdi()

✅ IGP-DI já está atualizado no Supabase. Último mês: 06/2026


## Importando as views do PdAdmin

In [81]:
consulta_conexao = text("""
    SELECT
        current_database() AS banco_atual,
        current_user AS usuario_atual,
        current_schema() AS schema_atual,
        current_setting('search_path') AS search_path,
        inet_server_addr() AS endereco_servidor,
        inet_server_port() AS porta_servidor;
""")

with engine.connect() as conexao:
    diagnostico_conexao = pd.read_sql_query(consulta_conexao, conexao)

display(diagnostico_conexao)

,banco_atual,usuario_atual,schema_atual,search_path,endereco_servidor,porta_servidor
0,postgres,lradmin,public,"""$user"", public",10.34.0.4,5432


#### Importação individual das views

In [ ]:
# Cada view recebe um DataFrame próprio para permitir tratamentos específicos.
df_revenue               = importar_view(nome_view="vw_revenue",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_cattle                = importar_view(nome_view="vw_cattle",                engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_expense               = importar_view(nome_view="vw_expense",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_feeding               = importar_view(nome_view="vw_feeding",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_labor                 = importar_view(nome_view="vw_labor",                 engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_own_milk              = importar_view(nome_view="vw_own_milk",              engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_area                  = importar_view(nome_view="vw_area_history",        engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_asset_payment_history = importar_view(nome_view="vw_asset_payment_history", engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")

Importando analytics_mart.vw_revenue...
Importando analytics_mart.vw_cattle...
Importando analytics_mart.vw_expense...
Importando analytics_mart.vw_feeding...
Importando analytics_mart.vw_labor...
Importando analytics_mart.vw_own_milk...
Importando analytics_mart.vw_area_ativa_mes...
Importando analytics_mart.vw_asset_payment_history...


In [ ]:
# ============================================================
# ATIVOS: CORRIGIR CADA ITEM E DEPOIS AGREGAR POR MÊS
# ============================================================

colunas_necessarias_ativos = [
    "id_property",
    "classification",
    "acquired_at",
    "reference_month",
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

colunas_ausentes_ativos = [
    coluna
    for coluna in colunas_necessarias_ativos
    if coluna not in df_asset_payment_history.columns
]

if colunas_ausentes_ativos:
    raise KeyError(
        "Colunas ausentes em df_asset_payment_history: "
        f"{colunas_ausentes_ativos}"
    )

df_assets = df_asset_payment_history.copy()

df_assets["acquisition_month"] = (
    pd.to_datetime(df_assets["acquired_at"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_assets["reference_month"] = (
    pd.to_datetime(df_assets["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Uma linha por mês na série do IGP-DI.
df_igpdi_assets = df_igpdi[["data", "igpdi"]].copy()
df_igpdi_assets["data"]  = (pd.to_datetime(df_igpdi_assets["data"], errors="coerce") .dt.to_period("M") .dt.to_timestamp())
df_igpdi_assets["igpdi"] = pd.to_numeric( df_igpdi_assets["igpdi"], errors="coerce", )
df_igpdi_assets = df_igpdi_assets.dropna(subset=["data", "igpdi"])

if df_igpdi_assets.duplicated("data").any():
    raise ValueError("Existem meses duplicados na série do IGP-DI.")

# Primeiro merge: índice do mês em que o ativo foi adquirido.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "acquisition_month", "igpdi": "igpdi_acquired"}),
    on="acquisition_month",
    how="left",
    validate="many_to_one",
)

# Segundo merge: índice do mês de referência da depreciação.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "reference_month", "igpdi": "igpdi_month"}),
    on="reference_month",
    how="left",
    validate="many_to_one",
)

sem_igpdi_ativo = (df_assets["igpdi_acquired"].isna() | df_assets["igpdi_month"].isna())

if sem_igpdi_ativo.any():
    print(f"⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em {sem_igpdi_ativo.sum():,} linhas." )

igpdi_valido = (df_assets["igpdi_acquired"].gt(0) & df_assets["igpdi_month"].gt(0) )

# Atualiza o valor da data de aquisição para o poder monetário do mês.
df_assets["asset_update_factor"] = 1.0
df_assets.loc[igpdi_valido, "asset_update_factor"] = ( df_assets.loc[igpdi_valido, "igpdi_month"] / df_assets.loc[igpdi_valido, "igpdi_acquired"] )

COLUNAS_MONETARIAS_ATIVOS = [
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

for coluna in COLUNAS_MONETARIAS_ATIVOS:
    df_assets[coluna] = (pd.to_numeric(df_assets[coluna], errors="coerce") * df_assets["asset_update_factor"])

# Normaliza acentos e capitalização para consolidar as classificações.
df_assets["classification_normalized"] = (
    df_assets["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

mascara_benfeitorias = (df_assets["classification_normalized"] == "benfeitorias")
mascara_maquinas = (df_assets["classification_normalized"] == "maquinas e equipamentos")

df_assets["monthly_depreciation_benfeitorias"]                     = (df_assets["monthly_depreciation"].where(mascara_benfeitorias, 0))
df_assets["monthly_depreciation_maquinas_e_equipamentos"]          = (df_assets["monthly_depreciation"].where(mascara_maquinas, 0))
df_assets["monthly_average_capital_stock_benfeitorias"]            = (df_assets["monthly_average_capital_stock"].where(mascara_benfeitorias, 0))
df_assets["monthly_average_capital_stock_maquinas_e_equipamentos"] = (df_assets["monthly_average_capital_stock"].where(mascara_maquinas, 0))

COLUNAS_MENSAIS_ATIVOS = [
    "monthly_depreciation_benfeitorias",
    "monthly_depreciation_maquinas_e_equipamentos",
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

df_asset_payment_history_monthly = (
    df_assets.loc[df_assets["reference_month"].ge(DATA_INICIAL) & df_assets["reference_month"].le(DATA_FINAL) ]
    .groupby(["id_property", "reference_month"], as_index=False)[ COLUNAS_MENSAIS_ATIVOS ]
    .sum()
    .sort_values(["id_property", "reference_month"])
    .reset_index(drop=True)
)

print( "Ativos corrigidos e agrupados: " f"{df_asset_payment_history_monthly.shape[0]:,} linhas mensais." )

⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em 27,323 linhas.
Ativos corrigidos e agrupados: 18,111 linhas mensais.


In [66]:
df_asset_payment_history_monthly

,id_property,reference_month,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2025-01-01,11371.936291,0.000000,175314.458414,0.000000
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2025-02-01,11485.637096,0.000000,177067.316908,0.000000
2,00220277-a58e-4b9e-9b89-e6acf8a1a400,2025-03-01,11428.253045,0.000000,176182.660714,0.000000
3,00220277-a58e-4b9e-9b89-e6acf8a1a400,2025-04-01,11462.185403,0.000000,176705.775947,0.000000
4,00220277-a58e-4b9e-9b89-e6acf8a1a400,2025-05-01,11364.744054,0.000000,175203.579923,0.000000
...,...,...,...,...,...,...
18106,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-03-01,13516.957377,22099.069283,326256.374397,143214.638785
18107,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-04-01,13843.288602,22632.592927,334132.972612,146672.177876
18108,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-05-01,13963.922839,22689.903737,337044.699544,147950.319710
18109,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-06-01,13854.085649,22511.429872,334393.578986,146786.574562


In [84]:
# Mantém compatibilidade com as funções de validação e merge existentes.
dados_views = {
    "vw_revenue": df_revenue,
    "vw_cattle": df_cattle,
    "vw_expense": df_expense,
    "vw_feeding": df_feeding,
    "vw_labor": df_labor,
    "vw_own_milk": df_own_milk,
    "vw_area_month": df_area,
    "vw_asset_payment_history": df_asset_payment_history_monthly,
}

for nome_view, df_view in dados_views.items():
    print(
        f"{nome_view}: {df_view.shape[0]:,} linhas e "
        f"{df_view.shape[1]:,} colunas."
    )

vw_revenue: 15,516 linhas e 20 colunas.
vw_cattle: 15,598 linhas e 18 colunas.
vw_expense: 15,730 linhas e 21 colunas.
vw_feeding: 14,794 linhas e 12 colunas.
vw_labor: 15,476 linhas e 6 colunas.
vw_own_milk: 12,728 linhas e 11 colunas.
vw_area_month: 35,094 linhas e 13 colunas.
vw_asset_payment_history: 18,111 linhas e 6 colunas.


In [ ]:
CHAVES = ["id_property", "reference_month"]

resumo_duplicidades = []
exemplos_duplicidades = {}

for nome_view, df_original in dados_views.items():

    print(f"Verificando {nome_view}...")

    # Trabalhar com uma cópia para não alterar o dado bruto
    df = df_original.copy()

    # ========================================================
    # TRATAMENTO DA VIEW DE ALIMENTAÇÃO
    # ========================================================
    if nome_view == "vw_feeding":

        # A coluna unit não será utilizada
        df = df.drop(
            columns=["unit"],
            errors="ignore",
        )
    
    # ========================================================
    # CONFERIR SE AS CHAVES EXISTEM
    # ========================================================

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:

        resumo_duplicidades.append({
            "view": nome_view,
            "linhas_totais": len(df),
            "linhas_em_chaves_duplicadas": None,
            "chaves_duplicadas": None,
            "chave_unica": False,
            "status": f"Chaves ausentes: {colunas_ausentes}",
        })

        print(f"Não foi possível verificar {nome_view}. Colunas ausentes: {colunas_ausentes}" )

        continue

    # ========================================================
    # PADRONIZAR AS CHAVES
    # ========================================================

    df["id_property"] = ( df["id_property"] .astype("string") .str.strip() )
    df["reference_month"] = ( pd.to_datetime( df["reference_month"], errors="coerce", ) .dt.to_period("M") .dt.to_timestamp() )

    # ========================================================
    # VERIFICAR DUPLICIDADES
    # ========================================================

    mascara_duplicada = df.duplicated( subset=CHAVES, keep=False, )

    df_duplicados = ( df.loc[mascara_duplicada] .sort_values(CHAVES) .copy() )
    quantidade_linhas_duplicadas = len( df_duplicados )
    quantidade_chaves_duplicadas = ( df_duplicados[CHAVES] .drop_duplicates() .shape[0] )

    resumo_duplicidades.append({
        "view": nome_view,
        "linhas_totais": len(df),
        "linhas_em_chaves_duplicadas": ( quantidade_linhas_duplicadas ),
        "chaves_duplicadas": ( quantidade_chaves_duplicadas ),
        "chave_unica": ( quantidade_linhas_duplicadas == 0 ),
        "status": ( "OK" if quantidade_linhas_duplicadas == 0 else "Possui duplicidades" ),
    })

    if quantidade_linhas_duplicadas > 0:
        exemplos_duplicidades[nome_view] = ( df_duplicados.head(20) )
        print( f"{nome_view}: " f"{quantidade_chaves_duplicadas:,} " "chaves duplicadas." )

    else:
        print( f"{nome_view}: nenhuma duplicidade." )


# ============================================================
# RESULTADO
# ============================================================

df_resumo_duplicidades = (
    pd.DataFrame(resumo_duplicidades)
    .sort_values(by=["chave_unica", "view"], ascending=[True, True])
    .reset_index(drop=True)
)

display(df_resumo_duplicidades)

Verificando vw_revenue...
vw_revenue: nenhuma duplicidade.
Verificando vw_cattle...
vw_cattle: nenhuma duplicidade.
Verificando vw_expense...
vw_expense: nenhuma duplicidade.
Verificando vw_feeding...
vw_feeding: nenhuma duplicidade.
Verificando vw_labor...
vw_labor: nenhuma duplicidade.
Verificando vw_own_milk...
vw_own_milk: nenhuma duplicidade.
Verificando vw_area_month...
vw_area_month: nenhuma duplicidade.
Verificando vw_asset_payment_history...
vw_asset_payment_history: nenhuma duplicidade.


,view,linhas_totais,linhas_em_chaves_duplicadas,chaves_duplicadas,chave_unica,status
0,vw_area_month,35094,0,0,True,OK
1,vw_asset_payment_history,18111,0,0,True,OK
2,vw_cattle,15598,0,0,True,OK
3,vw_expense,15730,0,0,True,OK
4,vw_feeding,14794,0,0,True,OK
5,vw_labor,15476,0,0,True,OK
6,vw_own_milk,12728,0,0,True,OK
7,vw_revenue,15516,0,0,True,OK


#### Tratando os dados antes de exportar

##### Funções de tratamento das views

In [86]:
def preparar_view(df: pd.DataFrame, nome_view: str) -> pd.DataFrame:
    """
    Prepara uma view mensal para os merges.

    - Padroniza id_property;
    - Padroniza o mês de referência;
    - Trata a view de área ativa;
    - Remove unit da view de alimentação;
    - Remove registros sem chave;
    - Filtra o período;
    - Ordena o resultado.
    """
    
    if df is None:
        raise ValueError( f"A view {nome_view} não foi importada." )

    # Trabalhar com uma cópia para preservar dados_views
    df = df.copy()
    
    # ========================================================
    # TRATAMENTO ESPECÍFICO DA ÁREA ATIVA
    # ========================================================
    if nome_view == "vw_area_ativa_mes":
        # Padronizar o nome da coluna mensal
        df = df.rename( columns={ "month_start": "reference_month"} )

 
    # ========================================================
    # TRATAMENTO ESPECÍFICO DA ALIMENTAÇÃO
    # ========================================================
    if nome_view == "vw_feeding":
        df = df.drop( columns=["unit"], errors="ignore")

    # ========================================================
    # CONFERIR AS CHAVES
    # ========================================================
    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise KeyError(f"A view {nome_view} não possui as colunas {colunas_ausentes}.")

    # ========================================================
    # PADRONIZAR ID_PROPERTY
    # ========================================================

    df["id_property"] = (
        df["id_property"]
        .astype("string")
        .str.strip()
    )

    # Transformar texto vazio em ausente
    df["id_property"] = df["id_property"].replace( "", pd.NA)

    # ========================================================
    # PADRONIZAR REFERENCE_MONTH
    # ========================================================

    df["reference_month"] = (
        pd.to_datetime(
            df["reference_month"],
            errors="coerce",
        )
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    # ========================================================
    # REMOVER LINHAS SEM CHAVE
    # ========================================================

    registros_sem_chave = ( df[CHAVES] .isna() .any(axis=1) )

    quantidade_sem_chave = int( registros_sem_chave.sum() )

    if quantidade_sem_chave > 0:

        print(
            f"{nome_view}: removendo "
            f"{quantidade_sem_chave:,} linhas sem chave."
        )

        df = df.loc[
            ~registros_sem_chave
        ].copy()

    # ========================================================
    # FILTRAR O PERÍODO
    # ========================================================

    df = df.loc[
        df["reference_month"].between(
            DATA_INICIAL,
            DATA_FINAL,
            inclusive="both",
        )
    ].copy()

    # ========================================================
    # ORGANIZAR O RESULTADO
    # ========================================================

    df = (
        df
        .sort_values(CHAVES)
        .reset_index(drop=True)
    )

    return df

def verificar_chave_unica(df: pd.DataFrame, nome_view: str) -> None:
    """
    Verifica se existe mais de uma linha para a mesma combinação
    de id_property e reference_month.

    O processamento é interrompido caso existam duplicidades.
    """

    # Identificar todas as linhas que fazem parte de chaves duplicadas
    mascara_duplicadas = df.duplicated(subset=CHAVES, keep=False)

    df_duplicadas = (df.loc[mascara_duplicadas] .copy())

    if not df_duplicadas.empty:

        quantidade_linhas_duplicadas = len(
            df_duplicadas
        )

        quantidade_chaves_duplicadas = (
            df_duplicadas[CHAVES]
            .drop_duplicates()
            .shape[0]
        )

        exemplos = (
            df_duplicadas[CHAVES]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"\nA view {nome_view} possui duplicidades.\n"
            f"Linhas envolvidas: "
            f"{quantidade_linhas_duplicadas:,}\n"
            f"Chaves duplicadas: "
            f"{quantidade_chaves_duplicadas:,}\n\n"
            f"Exemplos:\n"
            f"{exemplos.to_string(index=False)}"
        )

    print(
        f"{nome_view}: chave única confirmada "
        f"em {len(df):,} linhas."
    )

##### Criar DataFrame Final

In [87]:
# ============================================================
# PREPARAR TODAS AS VIEWS
# ============================================================

dados_preparados = {}

for nome_view, df_bruto in dados_views.items():
    
    print(f"\nPreparando {nome_view}...")

    df_preparado = preparar_view( df=df_bruto, nome_view=nome_view)
    
    verificar_chave_unica( df=df_preparado, nome_view=nome_view)

    dados_preparados[nome_view] = ( df_preparado.copy())

print("\nTodas as views foram preparadas.")

print("\nViews disponíveis em dados_preparados:")

for nome_view in dados_preparados:
    print(f"- {nome_view}")


Preparando vw_revenue...
vw_revenue: chave única confirmada em 10,569 linhas.

Preparando vw_cattle...
vw_cattle: chave única confirmada em 10,612 linhas.

Preparando vw_expense...
vw_expense: chave única confirmada em 10,657 linhas.

Preparando vw_feeding...
vw_feeding: chave única confirmada em 10,087 linhas.

Preparando vw_labor...
vw_labor: chave única confirmada em 10,543 linhas.

Preparando vw_own_milk...
vw_own_milk: chave única confirmada em 8,919 linhas.

Preparando vw_area_month...
vw_area_month: chave única confirmada em 19,063 linhas.

Preparando vw_asset_payment_history...
vw_asset_payment_history: chave única confirmada em 18,111 linhas.

Todas as views foram preparadas.

Views disponíveis em dados_preparados:
- vw_revenue
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- vw_area_month
- vw_asset_payment_history


In [88]:
df_final = (
    dados_preparados[VIEW_BASE]
    .copy()
    .sort_values(CHAVES)
    .reset_index(drop=True)
)

quantidade_linhas_base = len(df_final)

print(f"\nBase revenue criada com " f"{quantidade_linhas_base:,} linhas.")

print(f"Propriedades: " f"{df_final['id_property'].nunique():,}")

print(f"Período: " f"{df_final['reference_month'].min():%Y-%m} " f"a {df_final['reference_month'].max():%Y-%m}")


Base revenue criada com 10,569 linhas.
Propriedades: 948
Período: 2025-01 a 2026-07


##### Realizar Merges

In [89]:
# ============================================================
# MERGE DAS VIEWS COM A VW_REVENUE
# ============================================================

resumo_merge = []

for nome_view in VIEWS_MERGE:

    print(f"\nAdicionando {nome_view}...")

    # --------------------------------------------------------
    # 1. Conferir se a view foi preparada
    # --------------------------------------------------------

    if nome_view not in dados_preparados:
        raise KeyError(
            f"A view {nome_view} não foi encontrada "
            "em dados_preparados."
        )

    df_auxiliar = dados_preparados[nome_view].copy()

    # --------------------------------------------------------
    # 2. Conferir se as chaves existem
    # --------------------------------------------------------

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df_auxiliar.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            f"A view {nome_view} não possui as chaves "
            f"{colunas_ausentes}."
        )

    # --------------------------------------------------------
    # 3. Conferir novamente se a chave é única
    # --------------------------------------------------------

    duplicadas_auxiliar = df_auxiliar.duplicated(
        subset=CHAVES,
        keep=False,
    )

    if duplicadas_auxiliar.any():

        exemplos = (
            df_auxiliar.loc[
                duplicadas_auxiliar,
                CHAVES,
            ]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"A view {nome_view} possui duplicidades.\n\n"
            f"{exemplos.to_string(index=False)}"
        )

    # --------------------------------------------------------
    # 4. Renomear colunas que já existem no df_final
    # --------------------------------------------------------

    prefixo = nome_view.removeprefix("vw_")

    colunas_conflitantes = [
        coluna
        for coluna in df_auxiliar.columns
        if coluna not in CHAVES
        and coluna in df_final.columns
    ]

    if colunas_conflitantes:

        df_auxiliar = df_auxiliar.rename(
            columns={
                coluna: f"{prefixo}_{coluna}"
                for coluna in colunas_conflitantes
            }
        )

        print("Colunas renomeadas:", colunas_conflitantes)

    # --------------------------------------------------------
    # 5. Verificar a cobertura antes do merge
    # --------------------------------------------------------

    cobertura = (
        df_final[CHAVES]
        .merge(
            df_auxiliar[CHAVES],
            on=CHAVES,
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )

    chaves_encontradas = int(
        cobertura["_merge"]
        .eq("both")
        .sum()
    )

    chaves_sem_correspondencia = int(
        cobertura["_merge"]
        .eq("left_only")
        .sum()
    )

    linhas_antes = len(df_final)

    # --------------------------------------------------------
    # 6. Realizar o left merge
    # --------------------------------------------------------

    df_final = df_final.merge(
        df_auxiliar,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_final)

    # --------------------------------------------------------
    # 7. Validar se a quantidade de linhas foi preservada
    # --------------------------------------------------------

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge com {nome_view} alterou a quantidade "
            f"de linhas de {linhas_antes:,} para "
            f"{linhas_depois:,}."
        )

    # --------------------------------------------------------
    # 8. Registrar o resumo
    # --------------------------------------------------------

    cobertura_percentual = round(
        chaves_encontradas / linhas_antes * 100,
        2,
    )

    resumo_merge.append({
        "view": nome_view,
        "linhas_view": len(df_auxiliar),
        "chaves_encontradas": chaves_encontradas,
        "chaves_sem_correspondencia": (
            chaves_sem_correspondencia
        ),
        "cobertura_percentual": cobertura_percentual,
        "colunas_adicionadas": (
            len(df_auxiliar.columns)
            - len(CHAVES)
        ),
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois,
    })

    print( f"Correspondências: " f"{chaves_encontradas:,}" )
    print( f"Sem correspondência: " f"{chaves_sem_correspondencia:,}" )
    print( f"Cobertura: " f"{cobertura_percentual:.2f}%" )
    print( f"Linhas antes/depois: " f"{linhas_antes:,} / {linhas_depois:,}" )


# ============================================================
# ORGANIZAR O RESULTADO FINAL
# ============================================================

df_final = (
    df_final
    .sort_values(CHAVES)
    .reset_index(drop=True)
)


# ============================================================
# VALIDAÇÕES FINAIS
# ============================================================

assert len(df_final) == quantidade_linhas_base, (
    "Os merges alteraram a quantidade de linhas da revenue."
)

assert not df_final.duplicated(CHAVES).any(), (
    "A tabela final possui duplicidades por "
    "id_property e reference_month."
)


# ============================================================
# RESUMO DOS MERGES
# ============================================================

df_resumo_merge = pd.DataFrame(
    resumo_merge
)

print("\nTodos os merges foram concluídos com sucesso.")
print(f"Linhas da base revenue: {quantidade_linhas_base:,}" )
print(f"Linhas da tabela final: {df_final.shape[0]:,}" )
print(f"Colunas da tabela final: {df_final.shape[1]:,}" )
print(f"Duplicidades por propriedade e mês: {df_final.duplicated(CHAVES).sum()}")

display(df_resumo_merge)
display(df_final.head())


Adicionando vw_cattle...
Correspondências: 10,236
Sem correspondência: 333
Cobertura: 96.85%
Linhas antes/depois: 10,569 / 10,569

Adicionando vw_expense...
Correspondências: 10,372
Sem correspondência: 197
Cobertura: 98.14%
Linhas antes/depois: 10,569 / 10,569

Adicionando vw_feeding...
Correspondências: 9,904
Sem correspondência: 665
Cobertura: 93.71%
Linhas antes/depois: 10,569 / 10,569

Adicionando vw_labor...
Correspondências: 10,322
Sem correspondência: 247
Cobertura: 97.66%
Linhas antes/depois: 10,569 / 10,569

Adicionando vw_own_milk...
Colunas renomeadas: ['hired_labor_quantity', 'family_labor_quantity']
Correspondências: 8,849
Sem correspondência: 1,720
Cobertura: 83.73%
Linhas antes/depois: 10,569 / 10,569

Adicionando vw_area_month...
Correspondências: 10,433
Sem correspondência: 136
Cobertura: 98.71%
Linhas antes/depois: 10,569 / 10,569

Adicionando vw_asset_payment_history...
Correspondências: 10,324
Sem correspondência: 245
Cobertura: 97.68%
Linhas antes/depois: 10,569 

,view,linhas_view,chaves_encontradas,chaves_sem_correspondencia,cobertura_percentual,colunas_adicionadas,linhas_antes,linhas_depois
0,vw_cattle,10612,10236,333,96.85,16,10569,10569
1,vw_expense,10657,10372,197,98.14,19,10569,10569
2,vw_feeding,10087,9904,665,93.71,9,10569,10569
3,vw_labor,10543,10322,247,97.66,4,10569,10569
4,vw_own_milk,8919,8849,1720,83.73,9,10569,10569
5,vw_area_month,19063,10433,136,98.71,11,10569,10569
6,vw_asset_payment_history,18111,10324,245,97.68,4,10569,10569


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,hectares_atividade,hectares_area_total,land_value_propria,land_value_app_reserva,land_value_atividade,land_value_area_total,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,NaN,...,24.0,25.18,NaN,0.0,0.0,0.0,0.0,1215.722920,0.0,4028.860809
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,NaN,...,24.0,25.18,NaN,0.0,0.0,0.0,0.0,1205.387920,0.0,4499.193869
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,NaN,...,24.0,25.18,NaN,0.0,0.0,0.0,0.0,1183.632817,0.0,4913.467488
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,194327.56,69902.0,2.78,116.0,38.0,3.39,3.25,NaN,...,24.0,25.18,NaN,0.0,0.0,0.0,0.0,1182.855703,0.0,5405.392294
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,194675.56,71836.0,2.71,132.0,41.0,3.47,3.24,NaN,...,24.0,25.18,NaN,0.0,0.0,0.0,0.0,1185.270671,0.0,5912.589830


In [73]:
df_final

,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,received_loans,...,hectares_atividade,hectares_area_total,land_value_propria,land_value_app_reserva,land_value_atividade,land_value_area_total,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,0.0,...,24.00,25.18,NaN,0.0,0.000000,0.000000,0.000000,1215.722920,0.000000,4028.860809
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,0.0,...,24.00,25.18,NaN,0.0,0.000000,0.000000,0.000000,1205.387920,0.000000,4499.193869
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,0.0,...,24.00,25.18,NaN,0.0,0.000000,0.000000,0.000000,1183.632817,0.000000,4913.467488
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,194327.56,69902.0,2.78,116.0,38.0,3.39,3.25,0.0,...,24.00,25.18,NaN,0.0,0.000000,0.000000,0.000000,1182.855703,0.000000,5405.392294
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,194675.56,71836.0,2.71,132.0,41.0,3.47,3.24,0.0,...,24.00,25.18,NaN,0.0,0.000000,0.000000,0.000000,1185.270671,0.000000,5912.589830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10564,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-02-01,1897398.40,765080.0,2.48,170.0,18.0,3.79,3.31,0.0,...,207.54,266.98,153330.586561,80000.0,174332.658765,153330.586561,13364.231598,21849.375696,322570.059682,141596.481189
10565,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-03-01,2126922.40,765080.0,2.78,158.0,3.0,3.82,3.30,0.0,...,207.54,266.98,153330.586561,80000.0,174332.658765,153330.586561,13516.957377,22099.069283,326256.374397,143214.638785
10566,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-04-01,2311010.10,733654.0,3.15,167.0,9.0,4.04,3.33,0.0,...,207.54,266.98,153330.586561,80000.0,174332.658765,153330.586561,13843.288602,22632.592927,334132.972612,146672.177876
10567,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-05-01,2353731.83,751991.0,3.13,167.0,9.0,4.04,3.33,0.0,...,207.54,266.98,153330.586561,80000.0,174332.658765,153330.586561,13963.922839,22689.903737,337044.699544,147950.319710


In [74]:
df_final.columns.to_list()

['id_property',
 'reference_month',
 'milk_sold_revenue',
 'milk_volume_sold',
 'milk_unit_price',
 'ccs',
 'cpp',
 'fat',
 'protein',
 'received_loans',
 'animal_sale',
 'other_revenues',
 'price_bonus',
 'price_penalty',
 'milk_derivatives_revenue',
 'unit_price_derivative',
 'voluminous_sold',
 'concentrated_sold',
 'surplus_division',
 'lactating_cows',
 'dry_cows',
 'nursing',
 'rearing',
 'males',
 'other_categories',
 'total_cows',
 'total_cattle',
 'lactating_cows_value',
 'dry_cows_value',
 'nursing_value',
 'rearing_value',
 'males_value',
 'other_categories_value',
 'total_cows_value',
 'total_cattle_value',
 'general_expenses',
 'advance_payment',
 'administration',
 'land_lease',
 'technical_assistance',
 'animal_purchase',
 'land_purchase',
 'repairs',
 'loan_interest',
 'hormones',
 'taxes_fees',
 'medicines_vaccines',
 'bedding_replacement',
 'reproduction',
 'milk_replacer',
 'milking_material',
 'milk_calves',
 'energy',
 'fuel',
 'voluminous_purchased_quantity',
 'vo

##### Deflacionando as colunas ncessárias

In [90]:
# ============================================================
# DEFLACIONAR AS VARIÁVEIS MONETÁRIAS
# ============================================================

# A deflação ocorre antes do Feature Engineering.
df_integrada = df_final.copy()

# ============================================================
# PREPARAR A BASE DO IGP-DI
# ============================================================

# Seleciona somente as colunas necessárias.
df_igpdi_aux = df_igpdi[[ "data", "deflator"] ].copy()

# Padroniza a data do IGP-DI para o primeiro dia de cada mês.
df_igpdi_aux["data"] = (
    pd.to_datetime(df_igpdi_aux["data"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Converte o deflator para formato numérico.
df_igpdi_aux["deflator"] = pd.to_numeric(df_igpdi_aux["deflator"], errors="coerce")

# Remove linhas sem data válida.
df_igpdi_aux = df_igpdi_aux.dropna(subset=["data"])

# Verifica se existem meses duplicados na tabela do IGP-DI.
if df_igpdi_aux.duplicated("data").any():
    meses_duplicados = (
        df_igpdi_aux.loc[df_igpdi_aux.duplicated("data", keep=False), "data"]
        .dt.strftime("%Y-%m")
        .unique()
        .tolist()
    )
    raise ValueError( "Existem meses duplicados na base do IGP-DI: " f"{meses_duplicados}" )


# ============================================================
# PADRONIZAR O MÊS DA BASE INTEGRADA
# ============================================================

df_integrada["reference_month"] = (
    pd.to_datetime(df_integrada["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ============================================================
# ADICIONAR O DEFLATOR
# ============================================================

df_integrada = df_integrada.drop(
    columns=["data", "deflator"],
    errors="ignore",
)

df_integrada = df_integrada.merge(
    df_igpdi_aux,
    left_on="reference_month",
    right_on="data",
    how="left",
    validate="many_to_one",
)


# ============================================================
# VALIDAR O DEFLATOR
# ============================================================

meses_sem_deflator = (
    df_integrada.loc[ df_integrada["deflator"].isna(), "reference_month", ]
    .dropna()
    .dt.strftime("%Y-%m")
    .unique()
    .tolist()
)

if meses_sem_deflator:
    print(f"⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: {meses_sem_deflator}" )

# Na ausência de IGP-DI, mantém o valor nominal usando deflator igual a 1.
df_integrada["deflator"] = df_integrada["deflator"].fillna(1.0)

if df_integrada["deflator"].le(0).any():
    raise ValueError( "Foram encontrados valores de deflator iguais ou inferiores a zero." )

# ============================================================
# DEFINIR AS COLUNAS MONETÁRIAS DE RECEITA
# ============================================================

COLUNAS_DEFLACIONAR_RECEITAS = [
    "milk_sold_revenue",         # Receita total da venda de leite.
    "milk_unit_price",           # Preço unitário do leite.
    "received_loans",            # Empréstimos recebidos.
    "animal_sale",               # Receita com venda de animais.
    "other_revenues",            # Outras receitas.
    "price_bonus",               # Bonificação do preço do leite.
    "price_penalty",             # Penalização ou desconto aplicado ao leite.
    "milk_derivatives_revenue",  # Receita total com derivados.
    "unit_price_derivative",     # Preço unitário dos derivados.
    "voluminous_sold",           # Receita com venda de volumoso.
    "concentrated_sold",          
    "surplus_division",          # Receita com divisão de sobras.
]

COLUNAS_DEFLACIONAR_DESPESAS = [
    "general_expenses",
    "advance_payment",
    "administration",
    "land_lease",
    "technical_assistance",
    "animal_purchase",
    "land_purchase",
    "repairs",
    "loan_interest",
    "hormones",
    "taxes_fees",
    "medicines_vaccines",
    "bedding_replacement",
    "reproduction",
    "milk_replacer",
    "milking_material",
    "milk_calves",
    "energy",
    "fuel",
]


COLUNAS_DEFLACIONAR_ANIMAIS = [
    "lactating_cows_value",
    "dry_cows_value",
    "nursing_value",
    "rearing_value",
    "males_value",
    "other_categories_value",
]

COLUNAS_DEFLACIONAR = (
    COLUNAS_DEFLACIONAR_RECEITAS
    + COLUNAS_DEFLACIONAR_DESPESAS
    # + COLUNAS_DEFLACIONAR_ANIMAIS
)

# ============================================================
# CONFERIR SE AS COLUNAS EXISTEM
# ============================================================

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_DEFLACIONAR
    if coluna not in df_integrada.columns
]

if colunas_ausentes:
    raise KeyError(f"As seguintes colunas monetárias não foram encontradas: {colunas_ausentes}" )


# ============================================================
# DEFLACIONAR AS COLUNAS
# ============================================================

for coluna in COLUNAS_DEFLACIONAR:

    # Converte o valor monetário para número e substitui a própria coluna.
    df_integrada[coluna] = (
        pd.to_numeric(df_integrada[coluna], errors="coerce")
        / df_integrada["deflator"]
    )

# ============================================================
# ORGANIZAR A BASE
# ============================================================

# Remove a coluna auxiliar de data proveniente do IGP-DI. O deflator é mantido para rastreabilidade.
df_integrada = df_integrada.drop(columns=["data"], errors="ignore", )

# Substitui eventuais infinitos por NaN.
df_integrada[COLUNAS_DEFLACIONAR] = (
    df_integrada[COLUNAS_DEFLACIONAR]
    .replace( [np.inf, -np.inf], np.nan, )
)

# ============================================================
# CONFERÊNCIA
# ============================================================

display(
    df_integrada[
        [
            "id_property",
            "reference_month",
            "deflator",
            "milk_unit_price",
            "milk_sold_revenue",
            "general_expenses",
            "lactating_cows_value",
        ]
    ].head(20)
)

print("Deflação das receitas, despesas e valores dos animais concluída com sucesso.")

⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: ['2026-07']


,id_property,reference_month,deflator,milk_unit_price,milk_sold_revenue,general_expenses,lactating_cows_value
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,0.991500,3.166920,112986.205837,0.000000,0.0
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,0.983071,2.990629,164137.685128,2543.051919,0.0
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,0.965328,2.890209,162750.562487,0.000000,648000.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,0.964694,2.881742,201439.522025,0.000000,672000.0
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,0.966664,2.803456,201389.093719,728.071075,810000.0
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,0.970099,2.731680,189936.448857,110.813438,680000.0
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,0.969795,2.608798,179829.648529,0.000000,640000.0
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,0.969871,2.392071,145392.440798,0.000000,560000.0
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-12-01,0.970839,2.193978,119308.532178,0.000000,832000.0
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2026-01-01,0.972775,2.148493,144344.385074,0.000000,816000.0


Deflação das receitas, despesas e valores dos animais concluída com sucesso.


##### Feature Engineering

In [91]:
# ============================================================
# 1. UTILIZAR A BASE INTEGRADA JÁ DEFLACIONADA
# ============================================================

CHAVES = ["id_property", "reference_month"]

linhas_iniciais = len(df_integrada)

print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas antes das flags: {df_integrada.shape[1]:,}")


# ============================================================
# 2. MAPA DAS VIEWS E COLUNAS DE PRESENÇA
# ============================================================
MAPA_PRESENCA = {
    "vw_cattle": "has_cattle_data",
    "vw_expense": "has_expense_data",
    "vw_feeding": "has_feeding_data",
    "vw_labor": "has_labor_data",
    "vw_own_milk": "has_own_milk_data",
    "vw_asset_payment_history": "has_asset_data",
    "vw_area_month": "has_active_area_month",
    # "vw_dairy_production_system_monthly": "has_dairy_production_system_data",
}

# A revenue é a base da tabela final. Portanto, todas as linhas possuem revenue.
df_integrada["has_revenue_data"] = 1

# ============================================================
# 3. CRIAR AS FLAGS DE PRESENÇA
# ============================================================
for nome_view, nome_flag in MAPA_PRESENCA.items():

    print(f"Criando indicador de presença: {nome_flag}")

    if nome_view not in dados_preparados:
        raise KeyError( f"A view {nome_view} não foi encontrada em dados_preparados." )

    # Evita duplicação caso a célula seja executada novamente
    df_integrada = df_integrada.drop(
        columns=[nome_flag],
        errors="ignore",
    )

    # Uma linha por propriedade e mês presente na view
    chaves_disponiveis = (
        dados_preparados[nome_view][CHAVES]
        .dropna(subset=CHAVES)
        .drop_duplicates(subset=CHAVES)
        .assign(**{nome_flag: 1})
    )

    linhas_antes = len(df_integrada)

    df_integrada = df_integrada.merge(
        chaves_disponiveis,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_integrada)

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge da flag {nome_flag} alterou a quantidade de linhas: "
            f"{linhas_antes:,} para {linhas_depois:,}."
        )

    df_integrada[nome_flag] = (
        df_integrada[nome_flag]
        .fillna(0)
        .astype("int8")
    )

def somar_colunas_preservando_ausencia(df: pd.DataFrame, colunas: list[str]) -> pd.Series:
    """
    Soma as colunas informadas.
    Quando todas as colunas estiverem ausentes na linha, mantém o resultado como NaN em vez de transformar em zero.
    """

    colunas_ausentes = [
        coluna
        for coluna in colunas
        if coluna not in df.columns
    ]
    
    if colunas_ausentes:
        raise KeyError("Colunas necessárias não encontradas: " + ", ".join(colunas_ausentes))

    return df[colunas].sum(axis=1, min_count=1)

# ============================================================
# 4. ORGANIZAR AS COLUNAS DE PRESENÇA
# ============================================================

colunas_presenca = [
    "has_revenue_data",
    "has_cattle_data",
    "has_expense_data",
    "has_feeding_data",
    "has_labor_data",
    "has_own_milk_data",
    "has_asset_data",
    "has_active_area_month",
    # "has_dairy_production_system_data"
]


# ============================================================
# 5. VALIDAR O RESULTADO
# ============================================================

assert len(df_integrada) == linhas_iniciais, ("A criação das flags alterou a quantidade de linhas.")

assert not df_integrada.duplicated(CHAVES).any(), ("Foram geradas duplicidades por id_property e reference_month.")

print("\nFlags de presença criadas com sucesso.")
print(f"Linhas finais: {df_integrada.shape[0]:,}")
print(f"Colunas finais: {df_integrada.shape[1]:,}")


# ============================================================
# 6. CALCULAR A COBERTURA DE CADA VIEW
# ============================================================
df_cobertura_views = (
    df_integrada[colunas_presenca]
    .mean()
    .mul(100)
    .round(2)
    .rename("coverage_percentage")
    .rename_axis("source")
    .reset_index()
)

display(df_cobertura_views)

# ============================================================
# 7. CALCULAR INDICADORES PADRÃO
# ============================================================

# Renda do leite consumido
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['hired_labor_quantity_milk_revenue'] = df_integrada['own_milk_hired_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['family_labor_milk_revenue']         = df_integrada['own_milk_family_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['calves_quantity_milk_revenue']      = df_integrada['calves_quantity'] * df_integrada['milk_unit_price']

# Renda do leite
df_integrada['total_milk_revenue'] = df_integrada[[
    'milk_sold_revenue',
    'milk_derivatives_revenue',
    'price_bonus',
    'discarded_quantity_milk_revenue',
    'hired_labor_quantity_milk_revenue',
    'family_labor_milk_revenue',
    'calves_quantity_milk_revenue'
]].sum(axis=1).round(2) - df_integrada['price_penalty']

# Leite produzido
df_integrada['milk_produced'] = df_integrada[[
    'milk_volume_sold',
    'milk_volume_derivatives',
    'discarded_quantity',
    'own_milk_hired_labor_quantity',
    'own_milk_family_labor_quantity',
    'calves_quantity']].sum(axis=1).round(2)

# Renda da Atividade
df_integrada['total_activity_revenue'] = (
    df_integrada[[
        'total_milk_revenue',
        'received_loans',
        'animal_sale',
        'other_revenues',
        'voluminous_sold',
        'concentrated_sold',
        'surplus_division',
    ]].sum(axis=1)
)
# Preço do Leite
df_integrada['milk_revenue_liter'] = df_integrada['total_milk_revenue'] / df_integrada['milk_produced']

# Leite diário
# Quantidade de dias do mês
df_integrada["days_in_month"] = ( df_integrada["reference_month"].dt.days_in_month)

# Produção média diária de leite
df_integrada["milk_daily"] = (df_integrada["milk_produced"] / df_integrada["days_in_month"].replace(0, np.nan))

# Produção diária por vaca em lactação
df_integrada["milk_lactating_cow_day"] = (df_integrada["milk_daily"] / df_integrada["lactating_cows"].replace(0, np.nan))

# Vacas em lactação sobre o total de vacas
df_integrada["lactating_cows_total_cows"] = (df_integrada["lactating_cows"] / df_integrada["total_cows"].replace(0, np.nan)) * 100

# Vacas em lactação sobre o total do rebanho
df_integrada["lactating_cows_total_cattle"] = (df_integrada["lactating_cows"] / df_integrada["total_cattle"].replace(0, np.nan) ) * 100

# Ajustar mão de obra para dias/homem
df_integrada['hired_labor_quantity']  = df_integrada['hired_labor_quantity'] / df_integrada["days_in_month"]
df_integrada['family_labor_quantity'] = df_integrada['family_labor_quantity'] / df_integrada["days_in_month"]

# Quantidade total de mão de obra
df_integrada["total_labor_quantity"] = ( df_integrada[[ "hired_labor_quantity", "family_labor_quantity"]] .sum(axis=1, min_count=1) )

# Produção diária por unidade de mão de obra total
df_integrada["milk_total_labor_day"] = (df_integrada["milk_daily"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Vacas em lactação por unidade de mão de obra
df_integrada["lactating_cows_total_labor"] = (df_integrada["lactating_cows"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Custo de concentrado e minerais
df_integrada["concentrate_mineral_cost"] = (df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0))

# Custo total da alimentação
df_integrada["feeding_cost"] = (df_integrada["voluminous_amount_total"].fillna(0) + df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0) )

# Quantidade de concentrado 
df_integrada["concentrate_mineral_quantity"] = (df_integrada["concentrate_purchased_quantity"].fillna(0) + df_integrada["mineral_purchased_quantity"].fillna(0))

# Custo da alimentação por litro
df_integrada["feeding_cost_liter"] = (df_integrada["feeding_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de volumoso por litro
df_integrada["voluminous_cost_liter"] = (df_integrada["voluminous_amount_total"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de concentrado e minerais por litro
df_integrada["concentrate_mineral_cost_liter"] = (df_integrada["concentrate_mineral_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação do custo da alimentação no preço do leite
df_integrada["feeding_cost_milk_price"] = (df_integrada["feeding_cost_liter"] / df_integrada["milk_revenue_liter"].replace(0, np.nan) ) * 100

# Custo total da mão de obra
df_integrada["total_labor_expenses"] = (df_integrada[["hired_labor_expenses", "family_labor_expenses"]] .sum(axis=1, min_count=1))

# Custo da mão de obra contratada por litro
df_integrada["hired_labor_cost_liter"] = (df_integrada["hired_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo da mão de obra familiar por litro
df_integrada["family_labor_cost_liter"] = (df_integrada["family_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo total da mão de obra por litro
df_integrada["total_labor_cost_liter"] = (df_integrada["total_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação da mão de obra na receita do leite
df_integrada["labor_cost_milk_revenue"] = ( df_integrada["total_labor_expenses"] / df_integrada["total_milk_revenue"].replace(0, np.nan) ) * 100

# Agregação das demais despesas operacionais
COLUNAS_OUTRAS_DESPESAS = [
    "milk_replacer",
    "milking_material",
    "reproduction",
    "hormones",
    "medicines_vaccines",
    "technical_assistance",
    "taxes_fees",
    "land_lease",
    "repairs",
    "administration",
    "general_expenses",
    "bedding_replacement",
]

df_integrada["other_operating_expenses"] = (somar_colunas_preservando_ausencia(df=df_integrada, colunas=COLUNAS_OUTRAS_DESPESAS) )

# Estoque de capital de animais
df_integrada["animal_capital_stock"] = (df_integrada["lactating_cows"] * df_integrada["lactating_cows_value"]) + (df_integrada["dry_cows"] * df_integrada["dry_cows_value"]) + (df_integrada["nursing"] * df_integrada["nursing_value"]) + (df_integrada["rearing"] * df_integrada["rearing_value"]) + (df_integrada["males"] * df_integrada["males_value"]) + ((df_integrada["other_categories"] * df_integrada["other_categories_value"])/2)

# Estoque de capital da terra
df_integrada["land_capital_stock"] = df_integrada["hectares_propria"] * df_integrada["land_value_propria"]

# Define os componentes do estoque de capital fixo.
COLUNAS_CAPITAL_FIXO = [
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

# Soma o estoque médio de benfeitorias com o estoque médio de máquinas e equipamentos.
#
# min_count=2 exige que os dois valores estejam disponíveis.
df_integrada["fixed_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_FIXO]
    .sum(
        axis=1,
        min_count=len(COLUNAS_CAPITAL_FIXO),
    )
)


# Estoque de capital total
COLUNAS_CAPITAL_TOTAL = [
    "animal_capital_stock",
    "land_capital_stock",
    "fixed_capital_stock",
]

df_integrada["total_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_TOTAL]
    .sum(axis=1)
)

# Calcula o estoque de capital total por litro de produção diária.
#
# Esse indicador é o equivalente novo de:
# estoqueCapitalcomTerra_leiteDiario.
#
# A produção diária igual a zero é substituída por NaN
# para evitar divisão por zero.
df_integrada["total_capital_stock_milk_daily"] = (
    df_integrada["total_capital_stock"]
    / df_integrada["milk_daily"].replace(0, np.nan)
)


# Substitui eventuais resultados infinitos por NaN.
df_integrada[
    [
        "total_capital_stock",
        "total_capital_stock_milk_daily",
    ]
] = (
    df_integrada[
        [
            "total_capital_stock",
            "total_capital_stock_milk_daily",
        ]
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# INDICADORES DE ÁREA
# ============================================================

# Vacas em lactação por hectare de atividade
df_integrada["lactating_cows_hectare_activity"] = (df_integrada["lactating_cows"] / df_integrada["hectares_atividade"].replace(0, np.nan))

# Percentual da área arrendada
df_integrada["rented_area_percentage"] = (df_integrada["hectares_arrendada"] / df_integrada["hectares_area_total"].replace(0, np.nan) * 100)

df_integrada.head(20)

Linhas: 10,569
Colunas antes das flags: 93
Criando indicador de presença: has_cattle_data
Criando indicador de presença: has_expense_data
Criando indicador de presença: has_feeding_data
Criando indicador de presença: has_labor_data
Criando indicador de presença: has_own_milk_data
Criando indicador de presença: has_asset_data
Criando indicador de presença: has_active_area_month

Flags de presença criadas com sucesso.
Linhas finais: 10,569
Colunas finais: 101


,source,coverage_percentage
0,has_revenue_data,100.00
1,has_cattle_data,96.85
2,has_expense_data,98.14
3,has_feeding_data,93.71
4,has_labor_data,97.66
5,has_own_milk_data,83.73
6,has_asset_data,97.68
7,has_active_area_month,98.71


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,total_labor_cost_liter,labor_cost_milk_revenue,other_operating_expenses,animal_capital_stock,land_capital_stock,fixed_capital_stock,total_capital_stock,total_capital_stock_milk_daily,lactating_cows_hectare_activity,rented_area_percentage
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112986.205837,35677.00,3.166920,421.0,77.0,3.48,3.40,NaN,...,0.438294,13.839742,12023.332915,0.0,NaN,4028.860809,4.028861e+03,3.387780,3.500000,100.0
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,164137.685128,54884.00,2.990629,338.0,54.0,3.55,3.29,NaN,...,0.406337,13.587017,22924.127904,0.0,NaN,4499.193869,4.499194e+03,2.484237,3.250000,100.0
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,162750.562487,56311.00,2.890209,192.0,102.0,3.46,3.32,NaN,...,0.399343,13.817101,11558.215377,56326500.0,NaN,4913.467488,5.633141e+07,28464.105439,3.375000,100.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,201439.522025,69902.00,2.881742,116.0,38.0,3.39,3.25,NaN,...,0.255491,8.865863,37483.169287,60532000.0,NaN,5405.392294,6.053741e+07,26349.436511,3.500000,100.0
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,201389.093719,71836.00,2.803456,132.0,41.0,3.47,3.24,NaN,...,0.285228,10.174148,104447.516351,73724000.0,NaN,5912.589830,7.372991e+07,30201.206267,3.375000,100.0
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,189936.448857,69531.00,2.731680,112.0,3.0,3.51,3.20,NaN,...,0.250742,9.179044,48305.340294,61196000.0,NaN,5933.600496,6.120193e+07,24661.294113,3.541667,100.0
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,179829.648529,68932.00,2.608798,145.0,8.0,3.58,3.25,NaN,...,0.254288,9.747324,29787.727372,54959000.0,NaN,5931.743621,5.496493e+07,23215.337131,2.962963,100.0
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,145392.440798,60781.00,2.392071,137.0,3.0,3.50,3.27,NaN,...,0.299484,12.519860,68583.539669,44368000.0,NaN,6264.085087,4.437426e+07,21140.333210,2.592593,100.0
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-12-01,119308.532178,54380.00,2.193978,147.0,7.0,3.48,3.19,NaN,...,0.496506,22.630402,12937.096064,92332000.0,NaN,6270.338026,9.233827e+07,52638.587357,3.851852,100.0
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2026-01-01,144344.385074,67184.00,2.148493,250.0,8.0,3.44,3.29,NaN,...,0.288304,13.418880,29195.487085,88345000.0,NaN,6282.838530,8.835128e+07,37118.866033,3.777778,100.0


In [ ]:
df_integrada.columns.tolist()  

##### Importar Dimensão Produtor

In [92]:
from sqlalchemy import text

# ============================================================
# IMPORTAR APENAS AS COLUNAS DIMENSIONAIS NECESSÁRIAS
# ============================================================
consulta_dim_property = text(
    """
    SELECT
        id_property,
        property_name,
        labor_rural_code,
        entrepreneur_name,
        agroindustry_name,
        dairy_region,
        property_status
    FROM analytics_int.vw_dim_property_base;
    """
)

df_dim_property = pd.read_sql_query( consulta_dim_property, con=engine, )

# Padronizar a chave
df_dim_property["id_property"] = (df_dim_property["id_property"].astype("string").str.strip())

# Remover linhas sem chave
df_dim_property = (df_dim_property.dropna(subset=["id_property"]).reset_index(drop=True) )

# Validar uma linha por propriedade
if df_dim_property.duplicated("id_property").any():
    raise ValueError( "A dimensão possui mais de uma linha por id_property." )

# Torna o merge idempotente: remove versões anteriores das colunas dimensionais.
colunas_dimensionais = [
    coluna
    for coluna in df_dim_property.columns
    if coluna != "id_property"
]

colunas_dimensionais_antigas = [
    nome
    for coluna in colunas_dimensionais
    for nome in (coluna, f"{coluna}_x", f"{coluna}_y")
    if nome in df_integrada.columns
]

df_integrada = df_integrada.drop(
    columns=colunas_dimensionais_antigas,
    errors="ignore",
)

# Merge dimensional
linhas_antes = len(df_integrada)

df_integrada = df_integrada.merge(
    df_dim_property,
    on="id_property",
    how="left",
    validate="many_to_one",
)

if len(df_integrada) != linhas_antes:
    raise ValueError( "O merge dimensional alterou a quantidade de linhas." )

print("Merge dimensional concluído.")
print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas: {df_integrada.shape[1]:,}")

df_integrada.head()

Merge dimensional concluído.
Linhas: 10,569
Colunas: 142


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,total_capital_stock,total_capital_stock_milk_daily,lactating_cows_hectare_activity,rented_area_percentage,property_name,labor_rural_code,entrepreneur_name,agroindustry_name,dairy_region,property_status
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112986.205837,35677.0,3.166920,421.0,77.0,3.48,3.40,NaN,...,4.028861e+03,3.387780,3.500,100.0,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,164137.685128,54884.0,2.990629,338.0,54.0,3.55,3.29,NaN,...,4.499194e+03,2.484237,3.250,100.0,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,162750.562487,56311.0,2.890209,192.0,102.0,3.46,3.32,NaN,...,5.633141e+07,28464.105439,3.375,100.0,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,201439.522025,69902.0,2.881742,116.0,38.0,3.39,3.25,NaN,...,6.053741e+07,26349.436511,3.500,100.0,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,201389.093719,71836.0,2.803456,132.0,41.0,3.47,3.24,NaN,...,7.372991e+07,30201.206267,3.375,100.0,Fazenda São Bartolomeu,LR01964,Edgar Jose de Azevedo Neto,NESTLÉ,Ibiá - 1215,active_approved


##### Regras de consitência dos Indicadores Mensais

In [93]:
# ============================================================
# REGRAS MENSAIS DE CONSISTÊNCIA
# ============================================================
df_consistencia = df_integrada.copy(deep=True)

# ============================================================
# 1. DEFINIR AS COLUNAS NECESSÁRIAS
# ============================================================

# Lista das colunas que precisam existir antes de calcular os indicadores e as regras de consistência.
COLUNAS_NECESSARIAS_CONSISTENCIA = [
    "ccs",
    "cpp",
    "fat",
    "protein",
    "lactating_cows",
    "lactating_cows_total_cows",
    "lactating_cows_total_cattle",
    "milk_daily",
    "milk_total_labor_day",
    "lactating_cows_total_labor",
    "feeding_cost_milk_price",
    "voluminous_cost_liter",
    "concentrate_mineral_cost_liter",
    "hired_labor_cost_liter",
    # "total_capital_stock_milk_daily"
]


# Verifica quais colunas da lista acima não existem na df_consistencia.
colunas_ausentes = [
    coluna
    for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA
    if coluna not in df_consistencia.columns
]

# Interrompe a execução caso alguma coluna obrigatória esteja ausente.
if colunas_ausentes:
    raise KeyError( "As seguintes colunas necessárias para a consistência " f"não foram encontradas: {colunas_ausentes}" )


# ============================================================
# 2. GARANTIR QUE AS COLUNAS SEJAM NUMÉRICAS
# ============================================================

# Percorre cada coluna usada nas regras de consistência.
for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA:
    
    # Converte a coluna para número.
    # Valores que não puderem ser convertidos serão transformados em NaN.
    df_consistencia[coluna] = pd.to_numeric( df_consistencia[coluna], errors="coerce", )


# Substitui valores infinitos positivos e negativos por NaN.
# Isso impede que divisões inválidas sejam classificadas como consistentes.
df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA] = (
    df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA]
    .replace([np.inf, -np.inf], np.nan)
)

# ============================================================
# 4. FUNÇÃO PARA PADRONIZAR AS REGRAS
# ============================================================

def preparar_regra_consistencia(condicao: pd.Series) -> pd.Series:
    """
    Converte o resultado de uma regra em verdadeiro ou falso.

    Valores ausentes são classificados como False, ou seja,
    o critério não é considerado atendido quando não há informação.
    """

    # Substitui resultados ausentes por False.
    condicao = condicao.fillna(False)

    # Garante que o resultado final tenha tipo booleano.
    return condicao.astype(bool)


# ============================================================
# 5. QUALIDADE DO LEITE
# ============================================================

# CCS é considerada consistente quando for maior que 50.
df_consistencia["cons_ccs"] = preparar_regra_consistencia( df_consistencia["ccs"].gt(50) )

# CPP é considerada consistente quando for maior que 1.
df_consistencia["cons_cpp"] = preparar_regra_consistencia( df_consistencia["cpp"].gt(1) )

# Gordura é consistente quando estiver acima de 2,5 e abaixo de 5,5.
df_consistencia["cons_fat"] = preparar_regra_consistencia( df_consistencia["fat"].gt(2.5) & df_consistencia["fat"].lt(5.5) )

# Proteína é consistente quando estiver acima de 2,4 e abaixo de 4,5.
df_consistencia["cons_protein"] = preparar_regra_consistencia( df_consistencia["protein"].gt(2.4) & df_consistencia["protein"].lt(4.5) )


# ============================================================
# 6. ESTRUTURA DO REBANHO
# ============================================================

# Na base nova, lactating_cows_total_cows está em percentual. 
# O limite antigo de 0,20 a 0,99 corresponde agora a 20% a 99%.
df_consistencia["cons_lactating_cows_total_cows"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cows"].gt(20)
        & df_consistencia["lactating_cows_total_cows"].lt(99)
    )
)

# Na base nova, lactating_cows_total_cattle também está em percentual. 
# O limite antigo de 0,15 a 0,99 corresponde agora a 15% a 99%.
df_consistencia["cons_lactating_cows_total_cattle"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cattle"].gt(15)
        & df_consistencia["lactating_cows_total_cattle"].lt(99)
    )
)


# ============================================================
# 7. PRODUTIVIDADE E MÃO DE OBRA
# ============================================================

# A produção diária por vaca em lactação deve ser maior que 3 e menor que 45 litros por vaca por dia.
df_consistencia["cons_milk_lactating_cow_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_lactating_cow_day"].gt(3)
        & df_consistencia["milk_lactating_cow_day"].lt(45)
    )
)


# A produção diária por unidade de mão de obra deve ser maior que 20 e menor que 1.500 litros por trabalhador por dia.
df_consistencia["cons_milk_total_labor_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_total_labor_day"].gt(20)
        & df_consistencia["milk_total_labor_day"].lt(1500)
    )
)


# O número de vacas em lactação por unidade de mão de obra deve ser positivo e menor que 70.
# O limite inferior positivo evita que propriedades com zero vacas em lactação sejam classificadas como consistentes.
df_consistencia["cons_lactating_cows_total_labor"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_labor"].gt(0)
        & df_consistencia["lactating_cows_total_labor"].lt(70)
    )
)


# ============================================================
# 8. ALIMENTAÇÃO E CUSTOS
# ============================================================

# feeding_cost_milk_price está em percentual na base nova.
# O limite antigo de 0,15 a 1,50 corresponde a 15% a 150%.
df_consistencia["cons_feeding_cost_milk_price"] = (
    preparar_regra_consistencia(
        df_consistencia["feeding_cost_milk_price"].gt(15)
        & df_consistencia["feeding_cost_milk_price"].lt(150)
    )
)

# O custo do volumoso deve ser menor que R$ 3 por litro.
df_consistencia["cons_voluminous_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["voluminous_cost_liter"].lt(3)
    )
)

# O custo de concentrado e minerais deve ser maior que R$ 0,30 e menor que R$ 3,50 por litro.
df_consistencia["cons_concentrate_mineral_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["concentrate_mineral_cost_liter"].gt(0.3)
        & df_consistencia["concentrate_mineral_cost_liter"].lt(3.5)
    )
)

# O custo da mão de obra contratada deve ser menor que R$ 1 por litro.
df_consistencia["cons_hired_labor_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["hired_labor_cost_liter"].lt(1)
    )
)

# ============================================================
# 9. ESTOQUE DE CAPITAL
# ============================================================

# Verifica se o estoque total de capital por produção diária
# está abaixo do limite de consistência.
# df_consistencia["cons_total_capital_stock"] = (
#     preparar_regra_consistencia(
#         df_consistencia["total_capital_stock_milk_daily"].lt(30000)
#     )
# )

# ============================================================
# 10. MAPA DOS CRITÉRIOS
# ============================================================

# Relaciona cada coluna booleana ao nome que será exibido
# no relatório de critérios violados.
MAPA_CRITERIOS_CONSISTENCIA = {
    "cons_ccs": "CCS",
    "cons_cpp": "CPP",
    "cons_fat": "Gordura",
    "cons_protein": "Proteína",
    "cons_lactating_cows_total_cows": "VL/Total de vacas",
    "cons_lactating_cows_total_cattle": "VL/Rebanho total",
    "cons_milk_lactating_cow_day": "Produção/VL",
    "cons_milk_total_labor_day": "Produção/MDO",
    "cons_lactating_cows_total_labor": "VL/MDO",
    "cons_feeding_cost_milk_price": "Alimentação/Preço do leite",
    "cons_voluminous_cost_liter": "Custo de volumoso",
    "cons_concentrate_mineral_cost_liter": "Custo de concentrado",
    "cons_hired_labor_cost_liter": "Custo da MDO contratada",
    # "cons_total_capital_stock": "Estoque de capital fixo",
}


# Transforma as chaves do dicionário em uma lista.
# Essa lista contém todas as colunas de consistência.
colunas_criterios = list(
    MAPA_CRITERIOS_CONSISTENCIA.keys()
)


# ============================================================
# 11. CONTAR CRITÉRIOS ATENDIDOS E VIOLADOS
# ============================================================

# Soma os valores True de cada linha.
# No pandas, True equivale a 1 e False equivale a 0.
df_consistencia["total_consistency_criteria_ok"] = (
    df_consistencia[colunas_criterios]
    .sum(axis=1)
    .astype("int8")
)


# Salva a quantidade total de critérios avaliados.
df_consistencia["total_consistency_criteria"] = len(
    colunas_criterios
)


# Calcula quantos critérios não foram atendidos.
df_consistencia["total_consistency_criteria_violated"] = (
    df_consistencia["total_consistency_criteria"]
    - df_consistencia["total_consistency_criteria_ok"]
)


# ============================================================
# 12. CLASSIFICAÇÃO GERAL
# ============================================================

# Classifica como Consistente quando nenhum critério foi violado.
# Caso exista pelo menos uma violação, classifica como Inconsistente.
df_consistencia["consistency_status"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    "Consistente",
    "Inconsistente",
)


# Cria uma classificação numérica.
# Zero representa consistente e um representa inconsistente.
df_consistencia["consistency_id"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    0,
    1,
).astype("int8")


# ============================================================
# 13. LISTAR OS CRITÉRIOS VIOLADOS
# ============================================================

# Cria inicialmente uma coluna vazia.
df_consistencia["violated_consistency_criteria"] = ""


# Percorre cada critério e seu respectivo nome de exibição.
for coluna_criterio, nome_criterio in MAPA_CRITERIOS_CONSISTENCIA.items():

    # Identifica as linhas em que o critério não foi atendido.
    mascara_violacao = ~df_consistencia[coluna_criterio]

    # Adiciona o nome do critério à lista de violações da linha.
    df_consistencia.loc[
        mascara_violacao,
        "violated_consistency_criteria",
    ] += nome_criterio + "; "


# Remove o último ponto e vírgula e os espaços excedentes.
df_consistencia["violated_consistency_criteria"] = (
    df_consistencia["violated_consistency_criteria"]
    .str.rstrip("; ")
)


# Substitui textos vazios por "Nenhum".
# Isso acontece quando todos os critérios foram atendidos.
df_consistencia["violated_consistency_criteria"] = (
    df_consistencia["violated_consistency_criteria"]
    .replace("", "Nenhum")
)


# ============================================================
# 14. CONFERÊNCIA DO RESULTADO
# ============================================================

# Cria uma tabela resumida com a quantidade de registros
# consistentes e inconsistentes.
df_resumo_consistencia = (
    df_consistencia["consistency_status"]
    .value_counts(dropna=False)
    .rename_axis("consistency_status")
    .reset_index(name="records")
)


# Calcula o percentual de cada classificação.
df_resumo_consistencia["percentage"] = (
    df_resumo_consistencia["records"]
    .div(len(df_consistencia))
    .mul(100)
    .round(2)
)


# Exibe o resumo da classificação.
display(df_resumo_consistencia)


# Exibe uma amostra das principais colunas geradas.
display(
    df_consistencia[
        [
            "id_property",
            "reference_month",
            "consistency_status",
            "consistency_id",
            "total_consistency_criteria_ok",
            "total_consistency_criteria_violated",
            "violated_consistency_criteria",
        ]
    ]
    .head(20)
)

,consistency_status,records,percentage
0,Consistente,8497,80.4
1,Inconsistente,2072,19.6


,id_property,reference_month,consistency_status,consistency_id,total_consistency_criteria_ok,total_consistency_criteria_violated,violated_consistency_criteria
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,Consistente,0,13,0,Nenhum
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,Consistente,0,13,0,Nenhum
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,Consistente,0,13,0,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,Consistente,0,13,0,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,Consistente,0,13,0,Nenhum
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,Consistente,0,13,0,Nenhum
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,Consistente,0,13,0,Nenhum
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,Consistente,0,13,0,Nenhum
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-12-01,Consistente,0,13,0,Nenhum
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2026-01-01,Consistente,0,13,0,Nenhum


In [ ]:
df_consistencia.columns.tolist() 

### Preparar os dados para o Excel

In [94]:
def preparar_dataframe_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara um DataFrame para exportação pelo openpyxl.
    """

    df_excel = df.copy()

    # Excel não trabalha com infinito
    df_excel = df_excel.replace([np.inf, -np.inf], np.nan)

    # Excel não aceita datas com timezone
    for coluna in df_excel.columns:
        if isinstance( df_excel[coluna].dtype, pd.DatetimeTZDtype, ):
            df_excel[coluna] = ( df_excel[coluna] .dt.tz_localize(None) )
    
    return df_excel

#### Exportar uma planilha por view

In [96]:
# ============================================================
# EXPORTAR DF_INTEGRADA PARA EXCEL
# ============================================================

DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d")
PASTA_SAIDA = (Path.cwd().parent / "data" / "outputs" )
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
CAMINHO_ARQUIVO = (PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_mensais.xlsx")

# Criar uma cópia para não alterar a tabela original
df_exportacao = df_consistencia.copy()

# Remover valores infinitos
df_exportacao = df_exportacao.replace([np.inf, -np.inf], np.nan)

# Ordenar a tabela
df_exportacao = (df_exportacao.sort_values( ["id_property", "reference_month"] ) .reset_index(drop=True))

# Validar duplicidades
if df_exportacao.duplicated(["id_property", "reference_month"]).any():
    raise ValueError("Existem duplicidades por id_property e reference_month.")

# Preparar os dados para o Excel
df_exportacao = preparar_dataframe_para_excel(df_exportacao)

# Exportar com a formatação padrão
exportar_varias_abas_xlsx(abas={"Indicadores Mensais": df_exportacao }, caminho_saida=CAMINHO_ARQUIVO, fonte="Aptos")

print("Exportação concluída com sucesso.")
print(f"Linhas exportadas: {len(df_exportacao):,}")
print(f"Arquivo: {CAMINHO_ARQUIVO}")

Exportação concluída com sucesso.
Linhas exportadas: 10,569
Arquivo: c:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\2026_07_17_indicadores_mensais.xlsx
